🚀 [Run in JupyterLite](../lite/lab/index.html?path=lecture5.ipynb){target="_blank" .btn .btn-primary}

# Lecture 5: File I/O in Python

In this lecture, we'll learn how to read and write files in Python, focusing on common bioinformatics file formats: FASTA, FASTQ, and SAM.

# Reading and writing files in Python

First let's download a file we will be using:

::: {.callout-note}
## JupyterLite Compatibility
This notebook uses Python's `urllib.request` module instead of shell commands for JupyterLite compatibility. In a standard Jupyter environment, you could use:
```bash
!curl -sLO https://raw.githubusercontent.com/nekrut/BMMB554/master/2023/data/l9/mt_cds.fa
```
:::

In [ ]:
from urllib.request import urlretrieve
urlretrieve("https://raw.githubusercontent.com/nekrut/BMMB554/master/2023/data/l9/mt_cds.fa", "mt_cds.fa")

In Python, you can handle files using the built-in `open` function. The `open` function creates a file object, which you can use to read, write, or modify the file.

Here's an example of how to open a file for reading:

In [ ]:
f = open("mt_cds.fa", "r")

In this example, the open function takes two arguments: the name of the file, and the mode in which you want to open the file. The `r` mode indicates that you want to open the file for reading.

After you've opened the file, you can read its contents using the read method:

In [ ]:
contents = f.read()
print(contents[:500])  # Print first 500 characters

You can also read the file line by line using the readline method:

In [ ]:
with open("mt_cds.fa", "r") as f:
    line = f.readline()
    print(line)

The `with` statement automatically closes the file when you're done, which is the recommended way to handle files in Python. It ensures resources are properly released even if an error occurs.

You can also use the `with` statement to automatically close the file when you're done:

In [ ]:
with open("mt_cds.fa", "r") as f:
    contents = f.read()
    print(contents[:500])

You can also write to files using `write` method (note the `"w"` mode):

In [ ]:
with open("sample.txt", "w") as f:
    f.write("This is a new line.")

In [ ]:
# Verify what we wrote
with open("sample.txt", "r") as f:
    print(f.read())

If you open an existing file in write mode, its contents will be overwritten. If you want to append to an existing file instead, you can use the `"a"` mode:

In [ ]:
with open("sample.txt", "a") as f:
    f.write(" This is another line.")

# Verify
with open("sample.txt", "r") as f:
    print(f.read())

The `with` statement ensures files are properly closed even if an error occurs, making it the recommended approach for all file operations.

In addition to reading and writing text files, you can also use Python to handle binary files, such as images or audio files.

Let's download an image:

::: {.callout-note}
## JupyterLite Compatibility
This uses Python's `urllib.request` module for JupyterLite compatibility. In a standard Jupyter environment:
```bash
!curl -sLO https://imgs.xkcd.com/comics/file_extensions.png
```
:::

In [ ]:
from urllib.request import urlretrieve
urlretrieve("https://imgs.xkcd.com/comics/file_extensions.png", "file_extensions.png")

Here's an example of how to read an image file:

In [ ]:
with open("file_extensions.png", "rb") as f:
    contents = f.read()
    print(f"Read {len(contents)} bytes from image file")

Note that when working with binary files, you must use the `"rb"` mode for reading and the `"wb"` mode for writing.

There are many more features and methods related to file handling in Python, but the basics covered here should be enough to get you started.

---

## [FASTA](https://en.wikipedia.org/wiki/FASTA_format)

FASTA is a file format that is commonly used to store biological sequences, such as DNA or protein sequences. In Python, you can read a FASTA file by opening the file, reading the lines one by one, and processing the data as needed.

Here's an example of how you might read a FASTA file in Python:

In [ ]:
sequences = {}
with open("mt_cds.fa", "r") as file:
    header = ""
    sequence = ""
    for line in file:
        line = line.rstrip()
        if line.startswith('>'):
            if header != "":
                sequences[header] = sequence
                sequence = ""
            header = line[1:]
        else:
            sequence += line
    if header != "":
        sequences[header] = sequence

In [ ]:
# Let's see what we got
print(f"Number of sequences: {len(sequences)}")
print("\nHeaders:")
for header in list(sequences.keys())[:5]:  # Show first 5
    print(f"  {header[:60]}...")

In [ ]:
# Look at one sequence
first_header = list(sequences.keys())[0]
print(f"Header: {first_header}")
print(f"Sequence length: {len(sequences[first_header])}")
print(f"First 100 bp: {sequences[first_header][:100]}")

The code above uses a `with` statement to open the file and read the lines one by one. If a line starts with a `">"`, it is assumed to be a header, and the current sequence is stored in the dictionary using the current header as the key. If the line does not start with a `">"`, it is assumed to be part of the current sequence.

---

## [FASTQ](https://en.wikipedia.org/wiki/FASTQ_format)

Let's download a sample FASTQ file:

::: {.callout-note}
## JupyterLite Compatibility
This uses Python's `urllib.request` module for JupyterLite compatibility. In a standard Jupyter environment:
```bash
!curl -sLO https://raw.githubusercontent.com/nekrut/BMMB554/master/2023/data/l9/reads.fq
```
:::

In [ ]:
from urllib.request import urlretrieve
urlretrieve("https://raw.githubusercontent.com/nekrut/BMMB554/master/2023/data/l9/reads.fq", "reads.fq")

FASTQ is a file format that is commonly used to store high-throughput sequencing data. It consists of a series of records, each of which includes a header, a sequence, a quality score header, and a quality score string. In Python, you can read a FASTQ file by opening the file, reading the lines four at a time, and processing the data as needed.

Here's an example of how you might read a FASTQ file in Python:

In [ ]:
def read_fastq(file_path):
    records = []
    with open(file_path, "r") as f:
        while True:
            header = f.readline().strip()
            if header == "":
                break
            sequence = f.readline().strip()
            quality_header = f.readline().strip()
            quality = f.readline().strip()
            records.append((header, sequence, quality))
    return records

In this example, the `read_fastq` function takes a file path as an argument, and returns a list of records, where each record is a tuple of three strings: the header, the sequence, and the quality score string. The function uses a `while` loop to read the lines four at a time until the end of the file is reached.

You can use this function to read a FASTQ file like this:

In [ ]:
records = read_fastq("reads.fq")
print(f"Number of reads: {len(records)}")
print("\nFirst 3 records:")
for header, sequence, quality in records[:3]:
    print(f"Header: {header}")
    print(f"Sequence: {sequence[:50]}...")
    print(f"Quality: {quality[:50]}...")
    print()

You can modify the `read_fastq` function to process the data in any way you need.

---

## [SAM](https://en.wikipedia.org/wiki/SAM_(file_format))

Let's download an example SAM file:

::: {.callout-note}
## JupyterLite Compatibility
This uses Python's `urllib.request` module for JupyterLite compatibility. In a standard Jupyter environment:
```bash
!curl -sLO https://raw.githubusercontent.com/nekrut/BMMB554/master/2023/data/l9/sam_example.sam
```
:::

In [ ]:
from urllib.request import urlretrieve
urlretrieve("https://raw.githubusercontent.com/nekrut/BMMB554/master/2023/data/l9/sam_example.sam", "sam_example.sam")

SAM (Sequence Alignment/Map) is a file format that is used to store the results of DNA sequencing alignments. In Python, you can read a SAM file by opening the file, reading the lines one by one, and processing the data as needed.

Here's an example of how you might read a SAM file in Python:

In [ ]:
def read_sam(file_path):
    records = []
    with open(file_path, "r") as f:
        for line in f:
            if line.startswith("@"):
                continue  # Skip header lines
            fields = line.strip().split("\t")
            records.append(fields)
    return records

In this example, the `read_sam` function takes a file path as an argument, and returns a list of records, where each record is a list of fields. The function uses a `with` statement to open the file and read the lines one by one. If a line starts with an `"@"`, it is assumed to be a header and is ignored. If the line does not start with an `"@"`, it is assumed to be a record, and the fields are extracted by splitting the line on tabs.

You can use this function to read a SAM file like this:

In [ ]:
records = read_sam("sam_example.sam")
print(f"Number of alignment records: {len(records)}")
print("\nFirst record fields:")
for i, field in enumerate(records[0][:6]):  # Show first 6 fields
    print(f"  Field {i}: {field}")

The SAM format has 11 mandatory fields:

| Field | Description |
|-------|-------------|
| QNAME | Query template name |
| FLAG | Bitwise flag |
| RNAME | Reference sequence name |
| POS | 1-based leftmost mapping position |
| MAPQ | Mapping quality |
| CIGAR | CIGAR string |
| RNEXT | Reference name of mate/next read |
| PNEXT | Position of mate/next read |
| TLEN | Template length |
| SEQ | Segment sequence |
| QUAL | Quality scores |

You can modify the `read_sam` function to process the data in any way you need. For example, you might want to extract specific fields, such as the reference name, the start position, and the CIGAR string.

In [ ]:
# Example: Extract specific fields
for record in records[:5]:
    qname = record[0]
    flag = record[1]
    rname = record[2]
    pos = record[3]
    cigar = record[5]
    print(f"Read: {qname}, Ref: {rname}, Pos: {pos}, CIGAR: {cigar}")

---

## Writing files in bioinformatics formats

Now let's practice writing files. We'll write a simple FASTA file:

In [ ]:
def write_fasta(sequences, file_path):
    """Write sequences dictionary to a FASTA file.
    
    Args:
        sequences: dict with headers as keys and sequences as values
        file_path: output file path
    """
    with open(file_path, "w") as f:
        for header, sequence in sequences.items():
            f.write(f">{header}\n")
            # Write sequence in lines of 60 characters
            for i in range(0, len(sequence), 60):
                f.write(sequence[i:i+60] + "\n")

In [ ]:
# Create a small test dictionary
test_sequences = {
    "seq1 test sequence": "ATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG",
    "seq2 another sequence": "GCTAGCTAGCTAGCTAGCTAGCTAGCTA"
}

# Write to file
write_fasta(test_sequences, "test_output.fa")

# Verify
with open("test_output.fa", "r") as f:
    print(f.read())

## Summary

In this lecture, we covered:

1. **Basic file I/O**: Opening, reading, and writing files using `open()`, `read()`, `write()`, and the `with` statement
2. **FASTA format**: Reading and writing sequence files with headers starting with `>`
3. **FASTQ format**: Reading sequencing reads with quality scores (4 lines per record)
4. **SAM format**: Reading alignment files (tab-separated fields, headers start with `@`)

These are the foundational skills for handling bioinformatics data in Python!